### Import the necessary database

In [1]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [2]:
#In[2]:
# define function
import src.SAT_function_Obs_Fingerprint as data_process
import src.Data_Preprocess as preprosess

In [3]:
# import src.slurm_cluster as scluster
# client, scluster = scluster.init_dask_slurm_cluster(scale=4, cores=50, memory="200GB")

In [4]:
# read all segmented trend patterns (L=10..73) into a dict of DataArrays
import os
dir_ICV_seg_TREND = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/OBS_ICV_std/'
len_segments = np.arange(10, 74, 1)

ds = {}
for L in len_segments:
    fpath = os.path.join(dir_ICV_seg_TREND, f"OBS_ICV_MK_trend_segments_L{L}.nc")
    da = xr.open_dataset(fpath, chunks={"lat": 10, "lon": 10})['trend']
    ds[f"ICV_trend_{L}yr"] = da


### Calculate the regional mean of each trend Length 

In [5]:
lat = ds["ICV_trend_10yr"].lat
lon = ds["ICV_trend_10yr"].lon
# Extratropical South Pacific region
lat1 = 30
lat2 = 50
lon1 = 175
lon2 = 220

ds_NPI_masked = {}
for key in ds.keys():
    ds_NPI_masked[key] = data_process.selreg(ds[key],lat, lon, lat1, lat2, lon1, lon2)

In [6]:
lon

<xarray.DataArray 'lon' (lon: 180)>
array([  0.,   2.,   4.,   6.,   8.,  10.,  12.,  14.,  16.,  18.,  20.,  22.,
        24.,  26.,  28.,  30.,  32.,  34.,  36.,  38.,  40.,  42.,  44.,  46.,
        48.,  50.,  52.,  54.,  56.,  58.,  60.,  62.,  64.,  66.,  68.,  70.,
        72.,  74.,  76.,  78.,  80.,  82.,  84.,  86.,  88.,  90.,  92.,  94.,
        96.,  98., 100., 102., 104., 106., 108., 110., 112., 114., 116., 118.,
       120., 122., 124., 126., 128., 130., 132., 134., 136., 138., 140., 142.,
       144., 146., 148., 150., 152., 154., 156., 158., 160., 162., 164., 166.,
       168., 170., 172., 174., 176., 178., 180., 182., 184., 186., 188., 190.,
       192., 194., 196., 198., 200., 202., 204., 206., 208., 210., 212., 214.,
       216., 218., 220., 222., 224., 226., 228., 230., 232., 234., 236., 238.,
       240., 242., 244., 246., 248., 250., 252., 254., 256., 258., 260., 262.,
       264., 266., 268., 270., 272., 274., 276., 278., 280., 282., 284., 286.,
       288., 290., 292., 294., 296., 298., 300., 302., 304., 306., 308., 310.,
       312., 314., 316., 318., 320., 322., 324., 326., 328., 330., 332., 334.,
       336., 338., 340., 342., 344., 346., 348., 350., 352., 354., 356., 358.])
Coordinates:
  * lon      (lon) float64 0.0 2.0 4.0 6.0 8.0 ... 350.0 352.0 354.0 356.0 358.0
Attributes:
    standard_name:  longitude
    long_name:      longitude
    units:          degrees_east
    axis:           X

In [7]:
# ds_NPI_masked entries are tuples: (dataarray, lon, lat)
ds_NPI_masked["ICV_trend_73yr"][0].isel(segment=1)

<xarray.DataArray 'trend' (lat: 11, lon: 24)>
dask.array<getitem, shape=(11, 24), dtype=float64, chunksize=(10, 10), chunktype=numpy.ndarray>
Coordinates:
  * lat      (lat) float64 29.0 31.0 33.0 35.0 37.0 ... 41.0 43.0 45.0 47.0 49.0
  * lon      (lon) float64 174.0 176.0 178.0 180.0 ... 214.0 216.0 218.0 220.0

In [ ]:
def plot_trend(temp_data, lats, lons, levels=None, extend=None, cmap=None, 
                                 title="", ax=None, show_xticks=False, show_yticks=False):
    """
    Plot the trend spatial pattern using Robinson projection with significance overlaid.

    Parameters:
    - temp_data: 2D numpy array with the trend values.
    - lats, lons: 1D arrays of latitudes and longitudes.
    - p_values: 2D array with p-values for each grid point.
    - GMST_p_values: 2D array with GMST p-values for each grid point.
    - title: Title for the plot.
    - ax: Existing axis to plot on. If None, a new axis will be created.
    - show_xticks, show_yticks: Boolean flags to show x and y axis ticks.
    
    Returns:
    - contour_obj: The contour object from the plot.
    """
    # Plotting
    contour_obj = ax.contourf(lons, lats, temp_data, levels=levels, extend=extend, cmap=cmap, transform=ccrs.PlateCarree())

    ax.coastlines(resolution='110m')
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False,
                      color='gray', alpha=0.35, linestyle='--')

    # Disable labels on the top and right of the plot
    gl.top_labels = False
    gl.right_labels = False

    # Enable labels on the bottom and left of the plot
    gl.bottom_labels = show_xticks
    gl.left_labels = show_yticks
    gl.xformatter = cticker.LongitudeFormatter()
    gl.yformatter = cticker.LatitudeFormatter()
    gl.xlabel_style = {'size': 16}
    gl.ylabel_style = {'size': 16}
    
    if show_xticks:
        gl.bottom_labels = True
    if show_yticks:
        gl.left_labels = True
    
    # ax.set_title(title, loc='center', fontsize=18, pad=5.0)

    return contour_obj
# %%
plt.rcParams['figure.figsize'] = (8, 10)
plt.rcParams['font.size'] = 16
# plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.labelsize'] = 16
plt.rcParams['ytick.direction'] = 'out'
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['ytick.major.right'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['xtick.bottom'] = True
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['savefig.pad_inches'] = 0.1
plt.rcParams['savefig.transparent'] = True

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as mticker
import cartopy.feature as cfeature
import cartopy.mpl.ticker as cticker
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import seaborn as sns
from matplotlib.colors import ListedColormap
from matplotlib.colors import BoundaryNorm, ListedColormap
import cartopy.util as cutil
import seaborn as sns
import matplotlib.colors as mcolors
import palettable

In [ ]:
da_plot =ds["ICV_trend_10yr"].sel(segment=slice(1,10)).mean(dim='segment')

fig, ax = plt.subplots(subplot_kw={'projection': ccrs.Robinson()})

contour_obj =  plot_trend(da_plot, da_plot.lat, da_plot.lon, levels=np.arange(-0.5, 0.6, 0.1), extend='both', cmap='RdBu_r',
                                    title="SEP region SAT anomaly 2020", ax=ax, show_xticks=True, show_yticks=True)

# colorbar
cbar = plt.colorbar(contour_obj, ax=ax, orientation='horizontal', pad=0.05, aspect=50)
cbar.set_label('Temperature anomaly (°C)')
cbar.ax.tick_params(labelsize=14)

plt.show()

In [8]:
ds_NPI_masked

{'ICV_trend_10yr': (<xarray.DataArray 'trend' (segment: 164, lat: 11, lon: 24)>
  dask.array<getitem, shape=(164, 11, 24), dtype=float64, chunksize=(164, 10, 10), chunktype=numpy.ndarray>
  Coordinates:
    * lat      (lat) float64 29.0 31.0 33.0 35.0 37.0 ... 41.0 43.0 45.0 47.0 49.0
    * lon      (lon) float64 174.0 176.0 178.0 180.0 ... 214.0 216.0 218.0 220.0
  Dimensions without coordinates: segment,
  <xarray.DataArray 'lon' (lon: 24)>
  array([174., 176., 178., 180., 182., 184., 186., 188., 190., 192., 194., 196.,
         198., 200., 202., 204., 206., 208., 210., 212., 214., 216., 218., 220.])
  Coordinates:
    * lon      (lon) float64 174.0 176.0 178.0 180.0 ... 214.0 216.0 218.0 220.0
  Attributes:
      standard_name:  longitude
      long_name:      longitude
      units:          degrees_east
      axis:           X,
  <xarray.DataArray 'lat' (lat: 11)>
  array([29., 31., 33., 35., 37., 39., 41., 43., 45., 47., 49.])
  Coordinates:
    * lat      (lat) float64 29.0 31.0 

In [9]:
# calculate the weighted regional mean anomalies of trend patterns
ds_sel_NPI_masked_mean = {}
for key in ds_NPI_masked.keys():
    # ds_NPI_masked entries are tuples (dataarray, lon, lat) -> pass the dataarray to the weighted mean
    ds_sel_NPI_masked_mean[key] = data_process.calc_weighted_mean(ds_NPI_masked[key][0])

In [10]:
ds_sel_NPI_masked_mean

{'ICV_trend_10yr': <xarray.DataArray 'trend' (segment: 164)>
 dask.array<truediv, shape=(164,), dtype=float64, chunksize=(164,), chunktype=numpy.ndarray>
 Dimensions without coordinates: segment,
 'ICV_trend_11yr': <xarray.DataArray 'trend' (segment: 163)>
 dask.array<truediv, shape=(163,), dtype=float64, chunksize=(163,), chunktype=numpy.ndarray>
 Dimensions without coordinates: segment,
 'ICV_trend_12yr': <xarray.DataArray 'trend' (segment: 162)>
 dask.array<truediv, shape=(162,), dtype=float64, chunksize=(162,), chunktype=numpy.ndarray>
 Dimensions without coordinates: segment,
 'ICV_trend_13yr': <xarray.DataArray 'trend' (segment: 161)>
 dask.array<truediv, shape=(161,), dtype=float64, chunksize=(161,), chunktype=numpy.ndarray>
 Dimensions without coordinates: segment,
 'ICV_trend_14yr': <xarray.DataArray 'trend' (segment: 160)>
 dask.array<truediv, shape=(160,), dtype=float64, chunksize=(160,), chunktype=numpy.ndarray>
 Dimensions without coordinates: segment,
 'ICV_trend_15yr': <

### specify the 5 and 95 percentile values for each year step and output the 10-73yr unforced percentile timeseires for each regions

In [11]:
# define the function to calculate the percentile
def calc_percentile(da, q):
    """ Calculate the qth percentile of the data along the specified dimension.
    Args:
    da: xr.DataArray
    dim: str
    q: float
    Returns:
    xr.DataArray
    """
    # remove nans for da
    da = da.dropna(dim='segment')
    lower_percentile = np.percentile(da, q)
    upper_percentile = np.percentile(da, 100-q)
    
    return lower_percentile, upper_percentile

In [12]:
ds['ICV_trend_11yr']

<xarray.DataArray 'trend' (segment: 163, lat: 90, lon: 180)>
dask.array<open_dataset-a13757215e76170d6f332760606c3821trend, shape=(163, 90, 180), dtype=float64, chunksize=(163, 10, 10), chunktype=numpy.ndarray>
Coordinates:
  * lat      (lat) float64 -89.0 -87.0 -85.0 -83.0 -81.0 ... 83.0 85.0 87.0 89.0
  * lon      (lon) float64 0.0 2.0 4.0 6.0 8.0 ... 350.0 352.0 354.0 356.0 358.0
Dimensions without coordinates: segment

In [13]:
# calculate the regional mean's percentile
# 5%---[0]
unforced_trend_NPI_lower_percentile = {}

# 95%---[1]
unforced_trend_NPI_upper_percentile = {}

for key in ds_sel_NPI_masked_mean.keys():
    unforced_trend_NPI_lower_percentile[key], unforced_trend_NPI_upper_percentile[key] = calc_percentile(ds_sel_NPI_masked_mean[key], 5)
    

In [14]:
unforced_trend_NPI_lower_percentile

{'ICV_trend_10yr': -0.5729246955525278,
 'ICV_trend_11yr': -0.5145725040054819,
 'ICV_trend_12yr': -0.47041301109434414,
 'ICV_trend_13yr': -0.47350491619891394,
 'ICV_trend_14yr': -0.43140065315935694,
 'ICV_trend_15yr': -0.4096052714171157,
 'ICV_trend_16yr': -0.40494803947844393,
 'ICV_trend_17yr': -0.4055049863250877,
 'ICV_trend_18yr': -0.39944759080375936,
 'ICV_trend_19yr': -0.3850710477996595,
 'ICV_trend_20yr': -0.366865014010209,
 'ICV_trend_21yr': -0.35802314144979164,
 'ICV_trend_22yr': -0.3496174466765576,
 'ICV_trend_23yr': -0.349822557669085,
 'ICV_trend_24yr': -0.3462788594303226,
 'ICV_trend_25yr': -0.33632676188174826,
 'ICV_trend_26yr': -0.3156566962427861,
 'ICV_trend_27yr': -0.31064874147941246,
 'ICV_trend_28yr': -0.31152662925846286,
 'ICV_trend_29yr': -0.310075964006872,
 'ICV_trend_30yr': -0.2845098847004546,
 'ICV_trend_31yr': -0.2685713819470333,
 'ICV_trend_32yr': -0.25344160732328735,
 'ICV_trend_33yr': -0.24419346129227665,
 'ICV_trend_34yr': -0.2418452470

In [15]:
# check the seires for different time scales
unforced_trend_NPI_lower_percentile['ICV_trend_73yr'], unforced_trend_NPI_upper_percentile['ICV_trend_73yr']

(-0.07311212398606289, 0.10247847719165884)

In [ ]:
list(unforced_trend_NPI_lower_percentile.values())

In [16]:
# transform the dictionary to the dataarray
icv_years = list(unforced_trend_NPI_lower_percentile.keys())
unforced_trend_NPI_lower_percentile_da = xr.DataArray(
	list(unforced_trend_NPI_lower_percentile.values()),
	coords={'ICV_trend_year_lower': icv_years},
	dims=['ICV_trend_year_lower'],
)
unforced_trend_NPI_upper_percentile_da = xr.DataArray(
	list(unforced_trend_NPI_upper_percentile.values()),
	coords={'ICV_trend_year_upper': icv_years},
	dims=['ICV_trend_year_upper'],
)
    

In [17]:
unforced_trend_NPI_lower_percentile_da

<xarray.DataArray (ICV_trend_year_lower: 64)>
array([-0.5729247 , -0.5145725 , -0.47041301, -0.47350492, -0.43140065,
       -0.40960527, -0.40494804, -0.40550499, -0.39944759, -0.38507105,
       -0.36686501, -0.35802314, -0.34961745, -0.34982256, -0.34627886,
       -0.33632676, -0.3156567 , -0.31064874, -0.31152663, -0.31007596,
       -0.28450988, -0.26857138, -0.25344161, -0.24419346, -0.24184525,
       -0.24082513, -0.23298561, -0.23538345, -0.22734173, -0.22417213,
       -0.21373112, -0.19881668, -0.19186955, -0.18709296, -0.18444887,
       -0.17675362, -0.17193032, -0.16470184, -0.16029256, -0.15064587,
       -0.14610464, -0.14164707, -0.13965978, -0.13657031, -0.13440919,
       -0.13149331, -0.12804271, -0.12319585, -0.12140992, -0.11591933,
       -0.11413196, -0.11075999, -0.10670906, -0.10117243, -0.09928219,
       -0.09416896, -0.09044162, -0.08747816, -0.08519723, -0.08286835,
       -0.08205858, -0.07900751, -0.0739022 , -0.07311212])
Coordinates:
  * ICV_trend_year_lower  (ICV_trend_year_lower) <U14 'ICV_trend_10yr' ... 'I...

In [18]:
# save the percentile data
dir_out = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG5/data/percentile/'
os.makedirs(dir_out, exist_ok=True)
unforced_trend_NPI_lower_percentile_da.to_dataset(name="ICV_trend_lower").to_netcdf(dir_out+'internal_NPI_trend_lower_percentile.nc')
unforced_trend_NPI_upper_percentile_da.to_dataset(name="ICV_trend_upper").to_netcdf(dir_out+'internal_NPI_trend_upper_percentile.nc')